# Major Tournament Backtest: Wider Frozen Validation

The 2010-2022 World Cup audit used for model selection has only 256 matches, so
accuracy differences of a few matches between candidate models are within noise
(binomial standard error at 57% accuracy over 256 matches is about 3.1
percentage points). This notebook widens the frozen validation set to every
completed major-tournament edition since 2010: FIFA World Cups, UEFA Euros,
Copa America, Africa Cup of Nations, and AFC Asian Cups.

Protocol is identical to `12_world_cup_model_sweep_deep_learning.ipynb`: for
each tournament edition, every model is trained only on matches played before
that edition kicks off (from 2000 onward), then scored on all matches of the
edition. Editions are detected by clustering each tournament's match dates with
a 90-day gap rule, which correctly groups editions that span a year boundary
(for example AFCON 2025, played December 2025 to January 2026). The in-progress
2026 World Cup is excluded. No generated CSV reports are written.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'matches_with_elo.csv'
SEED = 20260707
LABELS = {'H': 0, 'D': 1, 'A': 2}
ORDERED = np.array(['H', 'D', 'A'])

matches = pd.read_csv(DATA, parse_dates=['date']).sort_values(['date', 'match_id']).reset_index(drop=True)
assert matches.date.max() >= pd.Timestamp('2026-07-03')
matches.shape

(49493, 22)

In [2]:
FAMILY_NAMES = {
    'FIFA World Cup': 'world_cup',
    'UEFA Euro': 'euro',
    'African Cup of Nations': 'afcon',
    'AFC Asian Cup': 'asian_cup',
}

def tournament_family(name):
    text = str(name)
    if 'qualif' in text.lower():
        return None
    if text.startswith('Copa Am'):
        return 'copa_america'
    return FAMILY_NAMES.get(text)

major = matches.assign(family=matches.tournament.map(tournament_family)).dropna(subset=['family']).copy()
major = major.sort_values(['family', 'date', 'match_id'], kind='mergesort')
gap_days = major.groupby('family').date.diff().dt.days.fillna(9999)
major['edition_id'] = (gap_days > 90).astype(int).groupby(major.family).cumsum()

editions = major.groupby(['family', 'edition_id']).agg(
    start=('date', 'min'), end=('date', 'max'), matches=('match_id', 'size')).reset_index()
editions = editions[(editions.start >= '2010-01-01') & (editions.end <= '2026-06-01') & (editions.matches >= 20)]
editions['label'] = editions.family + '_' + editions.start.dt.year.astype(str)
editions = editions.sort_values('start').reset_index(drop=True)

assert editions[editions.family == 'world_cup'].matches.eq(64).all()
assert len(editions[editions.family == 'world_cup']) == 4
display(editions)
print('folds:', len(editions), 'total test matches:', int(editions.matches.sum()))

,family,edition_id,start,end,matches,label
0,afcon,27,2010-01-10,2010-01-31,29,afcon_2010
1,world_cup,19,2010-06-11,2010-07-11,64,world_cup_2010
2,asian_cup,15,2011-01-07,2011-01-29,32,asian_cup_2011
3,copa_america,43,2011-07-01,2011-07-24,26,copa_america_2011
4,afcon,28,2012-01-21,2012-02-12,32,afcon_2012
5,euro,14,2012-06-08,2012-07-01,31,euro_2012
6,afcon,29,2013-01-19,2013-02-10,32,afcon_2013
7,world_cup,20,2014-06-12,2014-07-13,64,world_cup_2014
8,asian_cup,16,2015-01-09,2015-01-31,32,asian_cup_2015
9,afcon,30,2015-01-17,2015-02-08,32,afcon_2015


folds: 27 total test matches: 1141


In [3]:
def classify_tournament(name):
    text = str(name).lower()
    if 'qualif' in text:
        return 'qualifier'
    if str(name) == 'FIFA World Cup':
        return 'world_cup'
    markers = ['copa america', 'copa am\u00e9rica', 'african cup of nations', 'afc asian cup', 'uefa euro', 'european championship', 'gold cup', 'oceania', 'nations league']
    if any(marker in text for marker in markers):
        return 'continental'
    if text == 'friendly':
        return 'friendly'
    return 'other'

def add_world_cup_stage(frame):
    frame = frame.copy()
    frame['wc_match_number'] = -1
    mask = frame.tournament.eq('FIFA World Cup')
    frame.loc[mask, 'wc_match_number'] = frame.loc[mask].groupby(frame.loc[mask, 'date'].dt.year).cumcount()
    frame['wc_group_stage'] = frame.wc_match_number.between(0, 47).astype(float)
    frame['wc_knockout_stage'] = (frame.wc_match_number >= 48).astype(float)
    return frame

def build_team_panel(frame):
    points = {'H': 1.0, 'D': 0.5, 'A': 0.0}
    home = pd.DataFrame({
        'match_id': frame.match_id, 'date': frame.date, 'team': frame.home_team, 'is_home': True,
        'goals_for': frame.home_score.astype(float), 'goals_against': frame.away_score.astype(float),
        'team_elo': frame.home_elo.astype(float), 'result_points': frame.result.map(points).astype(float),
        'expected_points': frame.elo_home_probability + 0.5 * frame.elo_draw_probability,
    })
    away = pd.DataFrame({
        'match_id': frame.match_id, 'date': frame.date, 'team': frame.away_team, 'is_home': False,
        'goals_for': frame.away_score.astype(float), 'goals_against': frame.home_score.astype(float),
        'team_elo': frame.away_elo.astype(float), 'result_points': 1.0 - frame.result.map(points).astype(float),
        'expected_points': frame.elo_away_probability + 0.5 * frame.elo_draw_probability,
    })
    panel = pd.concat([home, away], ignore_index=True).sort_values(['team', 'date', 'match_id'], kind='mergesort').reset_index(drop=True)
    grouped = panel.groupby('team', sort=False)
    panel['days_since_match'] = grouped.date.diff().dt.days.fillna(365).clip(0, 365)
    recent_counts = pd.Series(0.0, index=panel.index)
    for _, idx in panel.groupby('team', sort=False).groups.items():
        dates = panel.loc[idx, 'date'].to_numpy()
        counts = []
        for pos, current in enumerate(dates):
            prior = dates[:pos]
            counts.append(float(np.sum((current - prior) <= np.timedelta64(365, 'D'))))
        recent_counts.loc[idx] = counts
    panel['recent_matches_365'] = recent_counts
    for window in [3, 5, 10]:
        for col in ['goals_for', 'goals_against', 'result_points']:
            fill = {'goals_for': 1.25, 'goals_against': 1.25, 'result_points': 0.5}[col]
            panel[f'rolling_{window}_{col}'] = grouped[col].transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean()).fillna(fill)
        actual = grouped.result_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        expected = grouped.expected_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        panel[f'rolling_{window}_form_vs_expected'] = (actual - expected).fillna(0.0)
        panel[f'rolling_{window}_elo_trend'] = (grouped.team_elo.shift(1) - grouped.team_elo.shift(1 + window)).fillna(0.0)
    return panel

def build_features(frame):
    frame = add_world_cup_stage(frame)
    frame['tournament_type'] = frame.tournament.map(classify_tournament)
    panel = build_team_panel(frame)
    value_cols = [c for c in panel.columns if c.startswith('rolling_')] + ['days_since_match', 'recent_matches_365']
    home = panel.loc[panel.is_home, ['match_id', *value_cols]].rename(columns={c: f'home_{c}' for c in value_cols})
    away = panel.loc[~panel.is_home, ['match_id', *value_cols]].rename(columns={c: f'away_{c}' for c in value_cols})
    out = frame.merge(home, on='match_id', how='left').merge(away, on='match_id', how='left')
    out['abs_elo_difference'] = out.elo_difference.abs()
    out['elo_similarity'] = np.exp(-out.abs_elo_difference / 200.0)
    out['elo_difference_sq'] = np.sign(out.elo_difference) * (out.elo_difference / 400.0) ** 2
    for window in [3, 5, 10]:
        out[f'recent_goal_total_{window}'] = out[f'home_rolling_{window}_goals_for'] + out[f'away_rolling_{window}_goals_for']
        out[f'recent_defence_total_{window}'] = out[f'home_rolling_{window}_goals_against'] + out[f'away_rolling_{window}_goals_against']
        out[f'recent_form_gap_{window}'] = out[f'home_rolling_{window}_result_points'] - out[f'away_rolling_{window}_result_points']
        out[f'recent_attack_gap_{window}'] = out[f'home_rolling_{window}_goals_for'] - out[f'away_rolling_{window}_goals_for']
        out[f'recent_defence_gap_{window}'] = out[f'home_rolling_{window}_goals_against'] - out[f'away_rolling_{window}_goals_against']
    out['rest_gap'] = out.home_days_since_match - out.away_days_since_match
    out['activity_gap_365'] = out.home_recent_matches_365 - out.away_recent_matches_365
    return out.sort_values(['date', 'match_id'], kind='mergesort').reset_index(drop=True)

featured = build_features(matches)
assert featured.filter(regex='rolling_|days_since|recent_|elo_similarity|stage|gap').isna().sum().sum() == 0
featured.shape

(49493, 80)

In [4]:
FEATURES = [
    'elo_difference', 'abs_elo_difference', 'elo_difference_sq', 'elo_similarity', 'neutral', 'home_elo', 'away_elo', 'wc_group_stage', 'wc_knockout_stage',
    'home_days_since_match', 'away_days_since_match', 'home_recent_matches_365', 'away_recent_matches_365', 'rest_gap', 'activity_gap_365',
]
for window in [3, 5, 10]:
    FEATURES += [
        f'home_rolling_{window}_goals_for', f'home_rolling_{window}_goals_against', f'home_rolling_{window}_result_points', f'home_rolling_{window}_form_vs_expected', f'home_rolling_{window}_elo_trend',
        f'away_rolling_{window}_goals_for', f'away_rolling_{window}_goals_against', f'away_rolling_{window}_result_points', f'away_rolling_{window}_form_vs_expected', f'away_rolling_{window}_elo_trend',
        f'recent_goal_total_{window}', f'recent_defence_total_{window}', f'recent_form_gap_{window}', f'recent_attack_gap_{window}', f'recent_defence_gap_{window}',
    ]
TOURNAMENT_LEVELS = ['world_cup', 'qualifier', 'continental', 'friendly']

def design(frame):
    out = frame[FEATURES].astype(float).copy()
    for col in [c for c in out.columns if 'elo' in c or c in ['home_elo', 'away_elo', 'elo_difference', 'abs_elo_difference']]:
        out[col] = out[col] / 400.0
    for col in ['home_days_since_match', 'away_days_since_match', 'rest_gap']:
        out[col] = out[col] / 30.0
    out['neutral'] = frame.neutral.astype(float)
    for level in TOURNAMENT_LEVELS:
        out[f'tournament_{level}'] = (frame.tournament_type == level).astype(float)
    return out

def labels_of(frame):
    return frame.result.map(LABELS).to_numpy()

def normalize(probs):
    probs = np.clip(np.asarray(probs, dtype=float), 1e-12, None)
    return probs / probs.sum(axis=1, keepdims=True)

def sample_weights(frame):
    age_years = (frame.date.max() - frame.date).dt.days / 365.25
    recency = np.exp(-age_years / 7.0)
    type_weight = frame.tournament_type.map({'world_cup': 1.7, 'qualifier': 1.25, 'continental': 1.15, 'friendly': 0.65, 'other': 1.0}).fillna(1.0)
    return np.asarray(recency * type_weight, dtype=float)

def brier(y, probs):
    return float(np.mean(np.sum((probs - np.eye(3)[y]) ** 2, axis=1)))

def predict_full_proba(model, x_test):
    raw = model.predict_proba(x_test)
    classes = getattr(model, 'classes_', None)
    if classes is None and hasattr(model, 'named_steps'):
        classes = model.named_steps[list(model.named_steps)[-1]].classes_
    out = np.zeros((len(x_test), 3), dtype=float)
    for source, cls in enumerate(classes):
        out[:, int(cls)] = raw[:, source]
    return normalize(out)

def fit_predict(name, train, test):
    x_train, x_test, y = design(train), design(test), labels_of(train)
    sw = sample_weights(train)
    if name == 'elo_reference':
        return normalize(test[['elo_home_probability', 'elo_draw_probability', 'elo_away_probability']].to_numpy(float))
    if name == 'logistic_l2_weighted':
        model = make_pipeline(StandardScaler(), LogisticRegression(C=0.25, max_iter=3000, random_state=SEED))
        model.fit(x_train, y, logisticregression__sample_weight=sw)
        return predict_full_proba(model, x_test)
    if name == 'logistic_l1_weighted':
        model = make_pipeline(StandardScaler(), LogisticRegression(C=0.08, penalty='l1', solver='saga', max_iter=2500, random_state=SEED))
        model.fit(x_train, y, logisticregression__sample_weight=sw)
        return predict_full_proba(model, x_test)
    if name == 'random_forest':
        model = RandomForestClassifier(n_estimators=180, max_depth=7, min_samples_leaf=24, random_state=SEED, n_jobs=-1, class_weight='balanced_subsample')
        model.fit(x_train, y, sample_weight=sw)
        return predict_full_proba(model, x_test)
    if name == 'extra_trees':
        model = ExtraTreesClassifier(n_estimators=220, max_depth=7, min_samples_leaf=24, random_state=SEED, n_jobs=-1, class_weight='balanced')
        model.fit(x_train, y, sample_weight=sw)
        return predict_full_proba(model, x_test)
    if name == 'hist_gradient_boosting':
        model = HistGradientBoostingClassifier(max_iter=110, learning_rate=0.04, max_leaf_nodes=13, l2_regularization=0.45, random_state=SEED)
        model.fit(x_train, y, sample_weight=sw)
        return predict_full_proba(model, x_test)
    if name == 'xgboost_regularized':
        model = XGBClassifier(objective='multi:softprob', num_class=3, n_estimators=220, max_depth=2, learning_rate=0.035, subsample=0.85, colsample_bytree=0.85, reg_lambda=2.5, reg_alpha=0.05, random_state=SEED, eval_metric='mlogloss', n_jobs=2)
        model.fit(x_train, y, sample_weight=sw)
        return normalize(model.predict_proba(x_test))
    if name == 'mlp_shallow':
        model = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(32,), activation='relu', alpha=0.03, learning_rate_init=0.001, early_stopping=True, validation_fraction=0.15, n_iter_no_change=8, max_iter=110, random_state=SEED))
        model.fit(x_train, y)
        return predict_full_proba(model, x_test)
    if name == 'mlp_deep':
        model = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32, 16), activation='relu', alpha=0.06, learning_rate_init=0.0008, early_stopping=True, validation_fraction=0.15, n_iter_no_change=8, max_iter=120, random_state=SEED))
        model.fit(x_train, y)
        return predict_full_proba(model, x_test)
    raise ValueError(name)

MODEL_NAMES = ['elo_reference', 'logistic_l2_weighted', 'logistic_l1_weighted', 'random_forest', 'extra_trees', 'hist_gradient_boosting', 'xgboost_regularized', 'mlp_shallow', 'mlp_deep']
ENSEMBLES = {
    'ensemble_boosted_linear_mlp': ['logistic_l2_weighted', 'hist_gradient_boosting', 'xgboost_regularized', 'mlp_deep'],
    'ensemble_tree_boosters': ['hist_gradient_boosting', 'xgboost_regularized'],
    'ensemble_all_non_elo': [name for name in MODEL_NAMES if name != 'elo_reference'],
}
display(design(featured.head()).shape)

(5, 64)

In [5]:
metric_rows = []
edition_match_ids = major.set_index(['family', 'edition_id']).match_id

for row in editions.itertuples():
    ids = major.loc[(major.family == row.family) & (major.edition_id == row.edition_id), 'match_id']
    test = featured[featured.match_id.isin(ids)].copy()
    assert len(test) == row.matches, row.label
    cutoff = test.date.min()
    train = featured[(featured.date < cutoff) & (featured.date >= '2000-01-01')].copy()
    y_test = labels_of(test)
    fold_probs = {}

    for name in MODEL_NAMES:
        fold_probs[name] = fit_predict(name, train, test)
    for ensemble_name, members in ENSEMBLES.items():
        fold_probs[ensemble_name] = normalize(np.mean([fold_probs[name] for name in members], axis=0))

    for name, probs in fold_probs.items():
        metric_rows.append({
            'edition': row.label, 'family': row.family, 'year': row.start.year, 'model': name, 'matches': len(test),
            'log_loss': log_loss(y_test, probs, labels=[0, 1, 2]), 'brier': brier(y_test, probs),
            'accuracy': float((probs.argmax(axis=1) == y_test).mean()),
            'correct': int((probs.argmax(axis=1) == y_test).sum()),
        })
    print(f'{row.label}: {len(test)} matches, train={len(train)}')

metrics = pd.DataFrame(metric_rows)
assert metrics.groupby('model').matches.sum().nunique() == 1
print('done:', metrics.matches.sum() // metrics.model.nunique(), 'test matches per model')

afcon_2010: 29 matches, train=9549


world_cup_2010: 64 matches, train=9802


asian_cup_2011: 32 matches, train=10399


copa_america_2011: 26 matches, train=10827


afcon_2012: 32 matches, train=11525


...

asian_cup_2024: 51 matches, train=22837


afcon_2024: 52 matches, train=22840


euro_2024: 51 matches, train=23309


copa_america_2024: 32 matches, train=23335


afcon_2025: 52 matches, train=24996
done: 1141 test matches per model


In [6]:
def aggregate_over(frame):
    return frame.groupby('model').apply(lambda g: pd.Series({
        'mean_log_loss': np.average(g.log_loss, weights=g.matches),
        'mean_brier': np.average(g.brier, weights=g.matches),
        'accuracy': g.correct.sum() / g.matches.sum(),
        'correct': int(g.correct.sum()),
        'matches': int(g.matches.sum()),
    }), include_groups=False).sort_values(['mean_log_loss', 'accuracy'], ascending=[True, False])

overall = aggregate_over(metrics)
per_family_accuracy = metrics.pivot_table(index='model', columns='family', values='correct', aggfunc='sum').div(
    metrics.pivot_table(index='model', columns='family', values='matches', aggfunc='sum'))
world_cup_only = aggregate_over(metrics[metrics.family == 'world_cup'])

total = int(overall.matches.iloc[0])
se = float(np.sqrt(0.55 * 0.45 / total))
print(f'binomial accuracy standard error at ~55% over {total} matches: +/-{se:.3%}')
display(overall.round(4))
display(per_family_accuracy.round(4))
display(world_cup_only.round(4))

binomial accuracy standard error at ~55% over 1141 matches: +/-1.473%


,mean_log_loss,mean_brier,accuracy,correct,matches
model,,,,,
logistic_l2_weighted,0.9670,0.5758,0.5504,628.0,1141.0
logistic_l1_weighted,0.9687,0.5772,0.5408,617.0,1141.0
ensemble_all_non_elo,0.9688,0.5782,0.5469,624.0,1141.0
ensemble_boosted_linear_mlp,0.9695,0.5785,0.5521,630.0,1141.0
xgboost_regularized,0.9724,0.5807,0.5364,612.0,1141.0
ensemble_tree_boosters,0.9738,0.5817,0.5346,610.0,1141.0
hist_gradient_boosting,0.9785,0.5845,0.5346,610.0,1141.0
elo_reference,0.9822,0.5856,0.5276,602.0,1141.0
mlp_deep,0.9956,0.5931,0.5337,609.0,1141.0


family,afcon,asian_cup,copa_america,euro,world_cup
model,,,,,
elo_reference,0.4822,0.6325,0.5294,0.5109,0.5352
ensemble_all_non_elo,0.4932,0.6386,0.5647,0.5272,0.5664
ensemble_boosted_linear_mlp,0.4959,0.6446,0.5588,0.5435,0.5742
ensemble_tree_boosters,0.4849,0.6386,0.5412,0.4946,0.5625
extra_trees,0.4466,0.5000,0.5412,0.3859,0.4648
hist_gradient_boosting,0.4849,0.6386,0.5412,0.4837,0.5703
logistic_l1_weighted,0.4904,0.6265,0.5529,0.5217,0.5625
logistic_l2_weighted,0.4959,0.6325,0.5529,0.5435,0.5781
mlp_deep,0.5014,0.6265,0.5471,0.5054,0.5312


,mean_log_loss,mean_brier,accuracy,correct,matches
model,,,,,
logistic_l2_weighted,0.9732,0.5691,0.5781,148.0,256.0
logistic_l1_weighted,0.9751,0.5718,0.5625,144.0,256.0
ensemble_boosted_linear_mlp,0.9773,0.5767,0.5742,147.0,256.0
xgboost_regularized,0.9822,0.5811,0.5586,143.0,256.0
ensemble_all_non_elo,0.9837,0.5820,0.5664,145.0,256.0
ensemble_tree_boosters,0.9845,0.5821,0.5625,144.0,256.0
elo_reference,0.9909,0.5884,0.5352,137.0,256.0
hist_gradient_boosting,0.9918,0.5853,0.5703,146.0,256.0
random_forest,1.0278,0.6169,0.4766,122.0,256.0


In [7]:
wc_l2 = world_cup_only.loc['logistic_l2_weighted']
assert int(wc_l2.matches) == 256
print(f"World-Cup-only sanity check vs notebook 12: logistic_l2_weighted {int(wc_l2.correct)}/256 accuracy={wc_l2.accuracy:.4f}")

best_ll = overall.index[0]
best_acc = overall.sort_values('accuracy', ascending=False).index[0]
print(f'Best log loss: {best_ll} ({overall.loc[best_ll].mean_log_loss:.4f})')
print(f'Best accuracy: {best_acc} ({overall.loc[best_acc].accuracy:.4f}, {int(overall.loc[best_acc].correct)}/{int(overall.loc[best_acc].matches)})')

lead = overall.sort_values('accuracy', ascending=False)
gap = lead.iloc[0].accuracy - lead.iloc[1].accuracy
print(f'Accuracy gap to runner-up: {gap:.3%} (noise threshold ~{2 * se:.3%} for two standard errors)')
assert np.isfinite(overall[['mean_log_loss', 'mean_brier', 'accuracy']].to_numpy()).all()
print('All wider backtest checks passed.')

World-Cup-only sanity check vs notebook 12: logistic_l2_weighted 148/256 accuracy=0.5781
Best log loss: logistic_l2_weighted (0.9670)
Best accuracy: ensemble_boosted_linear_mlp (0.5521, 630/1141)
Accuracy gap to runner-up: 0.175% (noise threshold ~2.946% for two standard errors)
All wider backtest checks passed.


## Interpretation

The wider validation set multiplies the frozen test sample roughly 4.5x compared
with the World-Cup-only audit, shrinking the binomial standard error on accuracy
from about 3.1 to about 1.5 percentage points. Model ranking on this set is the
one to trust for selection decisions; the World-Cup-only table is kept as a
sanity check against notebook 12. Accuracy gaps smaller than two standard errors
between candidate models should still be treated as ties, with mean log loss as
the tie-breaker because it uses the full probability vector rather than only the
argmax pick.